# Actividad 5 — Entrenamiento y Ajuste de Modelos de Clasificación de Sentimientos

**Alumno:** Alejandro Islas López  
**Fecha:** Junio 2025  
**Dataset:** Reseñas de restaurante — Zapopan, Jalisco (400 registros)  
**Tarea:** Clasificación multiclase (POS / NEU / NEG)  
**Modelos:** Regresión Logística vs Random Forest  
**Tracking:** MLflow (local)

## 0. Instalación de dependencias (Colab)

In [ ]:
# Ejecutar solo en Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install mlflow scikit-learn pandas matplotlib seaborn wordcloud -q
    # Clonar el repositorio
    !git clone https://github.com/TU_USUARIO/Actividad5.git
    %cd Actividad5

## 1. Importaciones y configuración global

In [ ]:
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from wordcloud import WordCloud

import mlflow
import mlflow.sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

CLASSES = [0, 1, 2]
CLASS_NAMES = ['NEG', 'NEU', 'POS']
LABEL_MAP   = {'POS': 2, 'NEU': 1, 'NEG': 0}

print('✅ Importaciones completadas')

## 2. EDA — Exploración del Dataset

In [ ]:
df_raw = pd.read_csv('../datos/datos_ini/resenas_restaurante_zapopan.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
print('=== Tipos de datos ===')
print(df_raw.dtypes)
print('\n=== Valores nulos ===')
print(df_raw.isnull().sum())
print('\n=== Estadísticas ===')
df_raw.describe(include='all')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de clases
conteos = df_raw['sentimiento'].value_counts()
axes[0].bar(conteos.index, conteos.values, color=['#e74c3c','#95a5a6','#2ecc71'])
axes[0].set_title('Distribución de Sentimientos')
axes[0].set_xlabel('Sentimiento')
axes[0].set_ylabel('Cantidad')
for i, (k, v) in enumerate(conteos.items()):
    axes[0].text(i, v + 1, f'{v} ({v/len(df_raw)*100:.1f}%)', ha='center')

# Longitud de comentarios
df_raw['longitud'] = df_raw['comentario'].str.len()
for sent, color in zip(['POS','NEU','NEG'], ['#2ecc71','#95a5a6','#e74c3c']):
    subset = df_raw[df_raw['sentimiento'] == sent]['longitud']
    axes[1].hist(subset, bins=20, alpha=0.6, label=sent, color=color)
axes[1].set_title('Distribución de Longitud de Comentarios')
axes[1].set_xlabel('Caracteres')
axes[1].legend()

plt.tight_layout()
plt.savefig('../datos/datos_limp/eda_distribucion.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sentimiento, color in zip(axes, ['POS','NEU','NEG'], ['Greens','Greys','Reds']):
    textos = ' '.join(df_raw[df_raw['sentimiento'] == sentimiento]['comentario'])
    wc = WordCloud(width=400, height=300, background_color='white',
                   colormap=color, max_words=50).generate(textos)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'Nube de Palabras — {sentimiento}')
    ax.axis('off')
plt.tight_layout()
plt.savefig('../datos/datos_limp/eda_wordclouds.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Limpieza y Preparación de Datos

In [ ]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df = df_raw.copy()
df = df.dropna(subset=['comentario', 'sentimiento'])
df['comentario_limpio'] = df['comentario'].astype(str).apply(limpiar_texto)
df['label'] = df['sentimiento'].map(LABEL_MAP)

df.to_csv('../datos/datos_limp/resenas_limpias.csv', index=False)
print(f'✅ Datos limpios guardados. Shape: {df.shape}')
df[['comentario', 'comentario_limpio', 'sentimiento', 'label']].head()

## 4. Vectorización TF-IDF y Split

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
X = vectorizer.fit_transform(df['comentario_limpio'])
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

with open('../datos/datos_limp/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print(f'Vocabulario TF-IDF: {len(vectorizer.vocabulary_)} términos')
print(f'Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras')

## 5. Configuración de MLflow

In [ ]:
mlflow.set_tracking_uri('file:///' + str(Path('../mlruns').resolve()))
mlflow.set_experiment('Actividad5_Sentimientos')
print(f'✅ MLflow configurado. URI: {mlflow.get_tracking_uri()}')

## 6. Funciones de Evaluación y Registro

In [ ]:
def calcular_metricas(y_true, y_pred, y_prob):
    return {
        'accuracy':        accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro':    recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro':        f1_score(y_true, y_pred, average='macro', zero_division=0),
        'roc_auc_ovr':     roc_auc_score(
            label_binarize(y_true, classes=CLASSES), y_prob,
            multi_class='ovr', average='macro'
        ),
    }

def plot_confusion_matrix(y_true, y_pred, nombre):
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False)
    ax.set_title(f'Matriz de Confusión — {nombre}')
    path = f'../datos/datos_limp/confusion_{nombre.lower().replace(" ","_")}.png'
    fig.savefig(path, bbox_inches='tight', dpi=120)
    plt.show()
    return path

def plot_roc_curves(y_true, y_prob, nombre):
    y_bin = label_binarize(y_true, classes=CLASSES)
    fig, ax = plt.subplots(figsize=(7, 5))
    for i, cls in enumerate(CLASS_NAMES):
        RocCurveDisplay.from_predictions(y_bin[:, i], y_prob[:, i], name=cls, ax=ax)
    ax.set_title(f'Curvas ROC (OVR) — {nombre}')
    path = f'../datos/datos_limp/roc_{nombre.lower().replace(" ","_")}.png'
    fig.savefig(path, bbox_inches='tight', dpi=120)
    plt.show()
    return path

print('✅ Funciones de evaluación cargadas')

## 7. Modelo 1 — Regresión Logística

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=SEED, multi_class='multinomial'),
    param_grid={'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'saga']},
    cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1
)
lr_grid.fit(X_train, y_train)

lr_best = lr_grid.best_estimator_
lr_cv_scores = cross_val_score(lr_best, X_train, y_train, cv=cv, scoring='f1_macro')

print(f'Mejores parámetros: {lr_grid.best_params_}')
print(f'CV F1-macro: {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}')

In [ ]:
lr_pred = lr_best.predict(X_test)
lr_prob = lr_best.predict_proba(X_test)
lr_metricas = calcular_metricas(y_test, lr_pred, lr_prob)

with mlflow.start_run(run_name='Regresion Logistica'):
    mlflow.log_params(lr_grid.best_params_)
    mlflow.log_param('cv_folds', 5)
    mlflow.log_param('vectorizer_ngram_range', '(1,2)')
    mlflow.log_param('vectorizer_max_features', 5000)
    mlflow.log_metrics(lr_metricas)
    mlflow.log_metric('cv_f1_macro_mean', lr_cv_scores.mean())
    mlflow.log_metric('cv_f1_macro_std',  lr_cv_scores.std())
    cm_path  = plot_confusion_matrix(y_test, lr_pred, 'Regresion Logistica')
    roc_path = plot_roc_curves(y_test, lr_prob, 'Regresion Logistica')
    mlflow.log_artifact(cm_path)
    mlflow.log_artifact(roc_path)
    mlflow.sklearn.log_model(lr_best, artifact_path='modelo')
    print('✅ Regresión Logística registrada en MLflow')

pd.DataFrame([lr_metricas], index=['Regresión Logística'])

## 8. Modelo 2 — Random Forest

In [ ]:
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_grid={
        'n_estimators':     [100, 200],
        'max_depth':        [None, 10, 20],
        'min_samples_split':[2, 5],
    },
    cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_
rf_cv_scores = cross_val_score(rf_best, X_train, y_train, cv=cv, scoring='f1_macro')

print(f'Mejores parámetros: {rf_grid.best_params_}')
print(f'CV F1-macro: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')

In [ ]:
rf_pred = rf_best.predict(X_test)
rf_prob = rf_best.predict_proba(X_test)
rf_metricas = calcular_metricas(y_test, rf_pred, rf_prob)

with mlflow.start_run(run_name='Random Forest'):
    mlflow.log_params(rf_grid.best_params_)
    mlflow.log_param('cv_folds', 5)
    mlflow.log_param('vectorizer_ngram_range', '(1,2)')
    mlflow.log_param('vectorizer_max_features', 5000)
    mlflow.log_metrics(rf_metricas)
    mlflow.log_metric('cv_f1_macro_mean', rf_cv_scores.mean())
    mlflow.log_metric('cv_f1_macro_std',  rf_cv_scores.std())
    cm_path  = plot_confusion_matrix(y_test, rf_pred, 'Random Forest')
    roc_path = plot_roc_curves(y_test, rf_prob, 'Random Forest')
    mlflow.log_artifact(cm_path)
    mlflow.log_artifact(roc_path)
    mlflow.sklearn.log_model(rf_best, artifact_path='modelo')
    print('✅ Random Forest registrado en MLflow')

pd.DataFrame([rf_metricas], index=['Random Forest'])

## 9. Comparativa Final de Modelos

In [ ]:
resultados = pd.DataFrame({
    'Regresión Logística': lr_metricas,
    'Random Forest':       rf_metricas,
}).T.round(4)

resultados['CV F1 mean'] = [lr_cv_scores.mean(), rf_cv_scores.mean()]
resultados['CV F1 std']  = [lr_cv_scores.std(),  rf_cv_scores.std()]
resultados

In [ ]:
metricas_plot = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']
x = np.arange(len(metricas_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, resultados.loc['Regresión Logística', metricas_plot], width, label='Regresión Logística', color='#3498db')
bars2 = ax.bar(x + width/2, resultados.loc['Random Forest',       metricas_plot], width, label='Random Forest',       color='#e67e22')

ax.set_xlabel('Métrica')
ax.set_ylabel('Valor')
ax.set_title('Comparativa de Modelos — Métricas en Test')
ax.set_xticks(x)
ax.set_xticklabels(['Accuracy','Precision','Recall','F1','ROC-AUC'])
ax.set_ylim(0, 1.1)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=8)

plt.tight_layout()
plt.savefig('../datos/datos_limp/comparativa_modelos.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Gráfica comparativa guardada')

## 10. Conclusiones

En esta celda se documentan las conclusiones después de ejecutar las celdas anteriores.

| Criterio | Regresión Logística | Random Forest |
|---|---|---|
| Velocidad de entrenamiento | Alta | Baja |
| Interpretabilidad | Alta | Media |
| F1-macro | Ver resultado | Ver resultado |
| ROC-AUC | Ver resultado | Ver resultado |

> **Modelo recomendado:** [completar después de ejecutar]